In [31]:
# CELL 01 - VERIFY PROJECT DIRECTORY

from pathlib import Path
import os

if Path.cwd().name != "additional_exercises":
    folder = Path.cwd() / "additional_exercises"

    if folder.is_dir():
        os.chdir(folder)
    else:
        raise RuntimeError("Open the additional_exercises folder first.")

print("Working directory:", Path.cwd())

assert Path("app.py").exists()
assert Path("templates").is_dir()

Working directory: c:\Users\Agawa\Documents\Lab10_Flask\additional_exercises


In [32]:
%%writefile models.py
# CELL 02 - UPDATED DATABASE MODELS

from flask_sqlalchemy import SQLAlchemy
from flask_login import UserMixin

db = SQLAlchemy()


class User(UserMixin, db.Model):

    __tablename__ = "users"

    id = db.Column(
        db.Integer, primary_key=True
    )

    email = db.Column(
        db.String(150),
        unique=True,
        nullable=False
    )

    password = db.Column(
        db.String(256),
        nullable=False
    )

    role = db.Column(
        db.String(20),
        nullable=False,
        default="viewer"
    )

    display_name = db.Column(
        db.String(100),
        default="Student"
    )

    image_filename = db.Column(
        db.String(255),
        nullable=True
    )

    def __repr__(self):
        return f"<User {self.email} - {self.role}>"


class Student(db.Model):

    __tablename__ = "students"

    id = db.Column(
        db.Integer, primary_key=True
    )

    full_name = db.Column(
        db.String(150),
        nullable=False
    )

    email = db.Column(
        db.String(150),
        unique=True,
        nullable=False
    )

    def __repr__(self):
        return f"<Student {self.full_name}>"

Overwriting models.py


In [33]:
# CELL 03 - ROLE-BASED ACCESS CONTROL

from pathlib import Path

path = Path("app.py")
code = path.read_text(encoding="utf-8")

for route in [
    "def add_student():",
    "def delete_student(id):"
]:

    replacement = route + '''

    if current_user.role != "admin":
        flash("Access denied. Admin permission required.")
        return redirect(url_for("students"))
'''

    if replacement not in code:

        if route not in code:
            raise RuntimeError(f"Route not found: {route}")

        code = code.replace(route, replacement, 1)

compile(code, "app.py", "exec")

path.write_text(code, encoding="utf-8")

print("Admin access control implemented!")

Admin access control implemented!


In [34]:
%%writefile templates/students.html
<!-- CELL 04 - ROLE-BASED STUDENT DASHBOARD -->

{% extends 'base.html' %}

{% block title %}Student Dashboard{% endblock %}

{% block content %}

<h2>Student Management Dashboard</h2>

<p>Welcome, {{ current_user.display_name }}!</p>

<p>
    Account Role:
    <strong>{{ current_user.role|upper }}</strong>
</p>

<p>Total Students: {{ students|length }}</p>

{% if current_user.role == 'admin' %}

    <h3>Add New Student</h3>

    <form method="POST"
          action="{{ url_for('add_student') }}">

        <input type="hidden"
               name="csrf_token"
               value="{{ csrf_token() }}">

        <input type="text"
               name="name"
               placeholder="Full Name"
               maxlength="150"
               required>

        <input type="email"
               name="email"
               placeholder="Email"
               maxlength="150"
               required>

        <button type="submit">
            Add Student
        </button>

    </form>

{% else %}

    <p>You have read-only access to student records.</p>

{% endif %}

<hr>

<h3>Student Records</h3>

<table class="student-table">

    <thead>

        <tr>
            <th>ID</th>
            <th>Full Name</th>
            <th>Email</th>

            {% if current_user.role == 'admin' %}
                <th>Action</th>
            {% endif %}
        </tr>

    </thead>

    <tbody>

    {% for student in students %}

        <tr>

            <td>{{ student.id }}</td>

            <td>{{ student.full_name }}</td>

            <td>{{ student.email }}</td>

            {% if current_user.role == 'admin' %}

                <td>

                    <form method="POST"
                          action="{{ url_for(
                              'delete_student',
                              id=student.id
                          ) }}">

                        <input type="hidden"
                               name="csrf_token"
                               value="{{ csrf_token() }}">

                        <button type="submit">
                            Delete
                        </button>

                    </form>

                </td>

            {% endif %}

        </tr>

    {% else %}

        <tr>
            <td colspan="4">No students found.</td>
        </tr>

    {% endfor %}

    </tbody>

</table>

{% endblock %}

Overwriting templates/students.html


In [35]:
# CELL 05 - DASHBOARD STYLING

from pathlib import Path

path = Path("static/css/style.css")

css = path.read_text(encoding="utf-8")

additional_css = """
/* Additional Exercise 1 */

.student-table {
    width: 100%;
    border-collapse: collapse;
    margin-top: 20px;
}

.student-table th,
.student-table td {
    border: 1px solid #cccccc;
    padding: 12px;
    text-align: left;
}

.student-table th {
    background-color: #44443B;
    color: white;
}

.student-table tr:nth-child(even) {
    background-color: #eeeeee;
}

.student-table tr:hover {
    background-color: #E6E4E0;
}
"""

if "/* Additional Exercise 1 */" not in css:
    css += "\n" + additional_css

path.write_text(css, encoding="utf-8")

print("Dashboard CSS added successfully!")

Dashboard CSS added successfully!


In [36]:
# CELL 06 - INITIALIZE SEPARATE DATABASE

from app import app
from models import db

with app.app_context():
    db.create_all()

print("Additional exercises database initialized.")
print("Database folder:", app.instance_path)

Additional exercises database initialized.
Database folder: c:\Users\Agawa\Documents\Lab10_Flask\additional_exercises\instance


In [37]:
# CELL 07 - CREATE TEST ACCOUNTS

from app import app
from models import db, User
from werkzeug.security import generate_password_hash
from getpass import getpass

accounts = [
    ("admin@example.com", "Admin", "admin"),
    ("viewer@example.com", "Viewer", "viewer")
]

with app.app_context():

    for email, name, role in accounts:

        existing = User.query.filter_by(
            email=email
        ).first()

        if existing:
            print(f"{role.upper()} account already exists.")
            continue

        password = getpass(
            f"Enter dummy password for {role}: "
        )

        if len(password) < 8:
            print("Use at least 8 characters. Run the cell again.")
            continue

        user = User(
            email=email,
            password=generate_password_hash(password),
            display_name=name,
            role=role
        )

        db.session.add(user)
        db.session.commit()

        print(f"{role.upper()} account created.")

print("Account setup finished.")

ADMIN account already exists.
VIEWER account already exists.
Account setup finished.


In [38]:
# CELL 08 - ADD SAMPLE STUDENTS

from app import app
from models import db, Student

sample_students = [
    ("Ana Reyes", "ana@example.com"),
    ("Juan Dela Cruz", "juan@example.com"),
    ("Maria Santos", "maria@example.com")
]

with app.app_context():

    for name, email in sample_students:

        existing = Student.query.filter_by(
            email=email
        ).first()

        if not existing:

            student = Student(
                full_name=name,
                email=email
            )

            db.session.add(student)

    db.session.commit()

    print("STUDENT RECORDS")

    for student in Student.query.order_by(
        Student.full_name
    ).all():

        print(
            f"{student.id}: "
            f"{student.full_name} "
            f"({student.email})"
        )
        

STUDENT RECORDS
1: Ana Reyes (ana@example.com)
2: Juan Dela Cruz (juan@example.com)
3: Maria Santos (maria@example.com)


In [39]:
# CELL 09 - CREATE IMAGE UPLOAD DIRECTORY

from pathlib import Path

upload_folder = Path("static/uploads")

upload_folder.mkdir(
    parents=True,
    exist_ok=True
)

print("Upload folder created!")
print(upload_folder.resolve())

Upload folder created!
C:\Users\Agawa\Documents\Lab10_Flask\additional_exercises\static\uploads


In [40]:
# CELL 10 - CREATE PROFILE FORM

from pathlib import Path

path = Path("forms.py")

code = path.read_text(encoding="utf-8")

profile_form = '''

# Additional Exercise 2

from flask_wtf.file import FileField, FileAllowed
from wtforms.validators import DataRequired


class ProfileForm(FlaskForm):

    display_name = StringField(
        "Display Name",
        validators=[
            DataRequired(),
            Length(min=2, max=100)
        ]
    )

    profile_picture = FileField(
        "Profile Picture",
        validators=[
            FileAllowed(
                ["jpg", "jpeg", "png"],
                "Only JPG, JPEG, and PNG images are allowed."
            )
        ]
    )

    submit = SubmitField("Update Profile")
'''

if "class ProfileForm" not in code:

    code += profile_form

compile(code, "forms.py", "exec")

path.write_text(code, encoding="utf-8")

print("Profile management form created!")

Profile management form created!


In [41]:
%%writefile templates/profile.html
<!-- CELL 11 - PROFILE MANAGEMENT TEMPLATE -->

{% extends 'base.html' %}

{% block title %}My Profile{% endblock %}

{% block content %}

<h2>My Profile</h2>

<div class="profile-card">

    {% if current_user.image_filename %}

        <img class="profile-image"
             src="{{ url_for(
                 'static',
                 filename='uploads/' +
                 current_user.image_filename
             ) }}"
             alt="Profile Picture">

    {% else %}

        <div class="profile-placeholder">
            No Profile Picture
        </div>

    {% endif %}

    <h3>{{ current_user.display_name }}</h3>

    <p>{{ current_user.email }}</p>

    <p>
        Role: {{ current_user.role|upper }}
    </p>

</div>

<hr>

<h3>Update Profile</h3>

<form method="POST"
      enctype="multipart/form-data">

    {{ form.hidden_tag() }}

    <div>

        {{ form.display_name.label }}

        <br>

        {{ form.display_name(size=35) }}

        {% for error in form.display_name.errors %}
            <p style="color:red;">{{ error }}</p>
        {% endfor %}

    </div>

    <div>

        {{ form.profile_picture.label }}

        <br>

        {{ form.profile_picture(
            accept=".jpg,.jpeg,.png"
        ) }}

        {% for error in form.profile_picture.errors %}
            <p style="color:red;">{{ error }}</p>
        {% endfor %}

        <small>
            Allowed formats: JPG, JPEG, PNG.
            Maximum upload size: 2 MB.
        </small>

    </div>

    <div>
        {{ form.submit() }}
    </div>

</form>

{% endblock %}

Overwriting templates/profile.html


In [42]:
# CELL 12 - IMPLEMENT PROFILE UPDATE ROUTE

from pathlib import Path

path = Path("app.py")

code = path.read_text(encoding="utf-8")

profile_route = '''

# ADDITIONAL EXERCISE 2 - PROFILE MANAGEMENT

from forms import ProfileForm
from werkzeug.utils import secure_filename
from PIL import Image, UnidentifiedImageError

app.config["MAX_CONTENT_LENGTH"] = 2 * 1024 * 1024

UPLOAD_FOLDER = Path(app.static_folder) / "uploads"

UPLOAD_FOLDER.mkdir(parents=True, exist_ok=True)


@app.route("/profile", methods=["GET", "POST"])
@login_required
def profile():

    form = ProfileForm()

    if request.method == "GET":
        form.display_name.data = current_user.display_name

    if form.validate_on_submit():

        image_file = form.profile_picture.data

        if image_file and image_file.filename:

            original_name = secure_filename(
                image_file.filename
            )

            extension = Path(
                original_name
            ).suffix.lower()

            allowed_extensions = {
                ".jpg", ".jpeg", ".png"
            }

            allowed_mime_types = {
                "image/jpeg", "image/png"
            }

            if (
                extension not in allowed_extensions
                or image_file.mimetype
                not in allowed_mime_types
            ):

                flash("Invalid image format.")

                return render_template(
                    "profile.html", form=form
                )

            try:

                with Image.open(image_file.stream) as image:

                    expected_format = (
                        "PNG" if extension == ".png"
                        else "JPEG"
                    )

                    if image.format != expected_format:
                        raise ValueError("Image format mismatch.")

                    if image.width * image.height > 16000000:
                        raise ValueError("Image dimensions too large.")

                    image.load()

                    image.thumbnail((1000, 1000))

                    filename = (
                        f"{current_user.id}_"
                        f"{secrets.token_hex(12)}"
                        ".png"
                    )

                    clean_image = Image.new("RGB", image.size)

                    clean_image.paste(image.convert("RGB"))

                    clean_image.save(
                        UPLOAD_FOLDER / filename,
                        format="PNG"
                    )

            except (
                UnidentifiedImageError,
                OSError,
                ValueError,
                Image.DecompressionBombError
            ):

                flash("The uploaded image is invalid.")

                return render_template(
                    "profile.html", form=form
                )

            current_user.image_filename = filename

        current_user.display_name = (
            form.display_name.data.strip()
        )

        db.session.commit()

        flash("Profile updated successfully!")

        return redirect(url_for("profile"))

    return render_template(
        "profile.html",
        form=form
    )


@app.errorhandler(413)
def file_too_large(e):

    return "Upload too large. Maximum request size is 2 MB.", 413

'''

marker = 'if __name__ == "__main__":'

if 'def profile():' not in code:

    if marker not in code:
        raise RuntimeError("Application entry point not found.")

    code = code.replace(
        marker,
        profile_route + "\n" + marker,
        1
    )

compile(code, "app.py", "exec")

path.write_text(code, encoding="utf-8")

print("Profile management route created!")

Profile management route created!


In [43]:
# CELL 13 - ADD PROFILE NAVIGATION

from pathlib import Path

path = Path("templates/base.html")

html = path.read_text(encoding="utf-8")

students_link = '<a href="{{ url_for(\'students\') }}">Students</a>'

profile_link = '<a href="{{ url_for(\'profile\') }}">My Profile</a>'

if profile_link not in html:

    if students_link not in html:
        raise RuntimeError("Students navigation link not found.")

    html = html.replace(
        students_link,
        students_link + "\n" + profile_link,
        1
    )

path.write_text(html, encoding="utf-8")

print("Profile navigation added!")

Profile navigation added!


In [44]:
# CELL 14 - PROFILE PAGE STYLING

from pathlib import Path

path = Path("static/css/style.css")

css = path.read_text(encoding="utf-8")

profile_css = """
/* Additional Exercise 2 */

.profile-card {
    text-align: center;
    background-color: white;
    padding: 25px;
    border-radius: 10px;
    border: 1px solid #cccccc;
    max-width: 450px;
    margin: 20px auto;
}

.profile-image {
    width: 150px;
    height: 150px;
    object-fit: cover;
    border-radius: 50%;
    border: 3px solid #9B806E;
}

.profile-placeholder {
    width: 150px;
    height: 150px;
    border-radius: 50%;
    background-color: #E6E4E0;
    display: flex;
    align-items: center;
    justify-content: center;
    margin: auto;
}

input[type="file"] {
    padding: 10px;
    background-color: white;
}
"""

if "/* Additional Exercise 2 */" not in css:

    css += "\n" + profile_css

path.write_text(css, encoding="utf-8")

print("Profile styling added successfully!")

Profile styling added successfully!


In [45]:
# CELL 15 - VERIFY APPLICATION FILES

from pathlib import Path

required_files = [
    "app.py",
    "models.py",
    "forms.py",
    "templates/base.html",
    "templates/students.html",
    "templates/profile.html",
    "static/css/style.css",
    "requirements.txt"
]

for file in required_files:

    if Path(file).exists():
        print(f"[OK] {file}")
    else:
        print(f"[MISSING] {file}")

for file in ["app.py", "models.py", "forms.py"]:

    compile(
        Path(file).read_text(encoding="utf-8"),
        file,
        "exec"
    )

print("\nPython syntax verification completed.")

[OK] app.py
[OK] models.py
[OK] forms.py
[OK] templates/base.html
[OK] templates/students.html
[OK] templates/profile.html
[OK] static/css/style.css
[OK] requirements.txt

Python syntax verification completed.


In [46]:
%%writefile static/css/style.css
/* OPTION A - MODERN ACADEMIC DASHBOARD */

:root {
    --navy: #1e3a5f;
    --navy-dark: #162c49;
    --blue: #2c5d8a;
    --light-blue: #eaf2fb;
    --white: #ffffff;
    --gray-bg: #f4f7fb;
    --gray-border: #d8e1ec;
    --text: #233142;
    --muted: #6b7a8c;
    --success: #2e8b57;
    --danger: #c0392b;
    --warning: #f39c12;
    --shadow: 0 8px 24px rgba(0, 0, 0, 0.08);
    --radius: 14px;
}

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    font-family: "Segoe UI", Tahoma, Geneva, Verdana, sans-serif;
    background: var(--gray-bg);
    color: var(--text);
}

a {
    text-decoration: none;
    color: inherit;
}

.layout {
    display: flex;
    min-height: 100vh;
}

.sidebar {
    width: 250px;
    background: linear-gradient(180deg, var(--navy), var(--navy-dark));
    color: var(--white);
    padding: 24px 18px;
    position: sticky;
    top: 0;
    height: 100vh;
}

.sidebar-brand {
    text-align: center;
    margin-bottom: 28px;
}

.sidebar-brand img {
    width: 72px;
    height: 72px;
    object-fit: contain;
    margin-bottom: 10px;
}

.sidebar-brand h2 {
    margin: 0;
    font-size: 20px;
}

.sidebar-brand p {
    margin: 6px 0 0;
    font-size: 13px;
    color: #d6e3f3;
}

.sidebar-nav {
    display: flex;
    flex-direction: column;
    gap: 10px;
}

.sidebar-nav a,
.sidebar-nav button {
    display: block;
    width: 100%;
    padding: 12px 14px;
    border-radius: 10px;
    border: none;
    background: transparent;
    color: var(--white);
    text-align: left;
    font-size: 15px;
    cursor: pointer;
    transition: 0.2s ease;
}

.sidebar-nav a:hover,
.sidebar-nav button:hover {
    background: rgba(255, 255, 255, 0.12);
}

.sidebar-user {
    margin-top: 24px;
    padding: 14px;
    border-radius: 12px;
    background: rgba(255, 255, 255, 0.10);
    font-size: 14px;
}

.sidebar-user strong {
    display: block;
    margin-bottom: 4px;
}

.main-area {
    flex: 1;
    padding: 28px;
}

.topbar {
    background: var(--white);
    border-radius: var(--radius);
    box-shadow: var(--shadow);
    padding: 18px 22px;
    margin-bottom: 22px;
    display: flex;
    justify-content: space-between;
    align-items: center;
}

.topbar h1 {
    margin: 0;
    font-size: 26px;
    color: var(--navy);
}

.topbar p {
    margin: 4px 0 0;
    color: var(--muted);
    font-size: 14px;
}

.card {
    background: var(--white);
    border-radius: var(--radius);
    box-shadow: var(--shadow);
    padding: 22px;
    margin-bottom: 22px;
}

.card h2,
.card h3 {
    margin-top: 0;
    color: var(--navy);
}

.grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
    gap: 18px;
}

.stat-card {
    background: var(--white);
    border-radius: var(--radius);
    box-shadow: var(--shadow);
    padding: 20px;
    border-left: 6px solid var(--blue);
}

.stat-card h3 {
    margin: 0 0 8px;
    font-size: 15px;
    color: var(--muted);
}

.stat-card p {
    margin: 0;
    font-size: 28px;
    font-weight: bold;
    color: var(--navy);
}

.role-badge {
    display: inline-block;
    padding: 6px 12px;
    border-radius: 999px;
    font-size: 12px;
    font-weight: bold;
    letter-spacing: 0.5px;
}

.role-admin {
    background: #d6ecff;
    color: #0b4f8a;
}

.role-viewer {
    background: #eaf5ea;
    color: #2e7d32;
}

.flash {
    padding: 12px 14px;
    border-radius: 10px;
    margin-bottom: 14px;
    font-size: 14px;
    background: #fdecea;
    color: var(--danger);
    border: 1px solid #f5c6cb;
}

form div {
    margin-bottom: 14px;
}

label {
    display: block;
    font-weight: 600;
    margin-bottom: 6px;
    color: var(--navy);
}

input[type="text"],
input[type="email"],
input[type="password"],
input[type="file"] {
    width: 100%;
    padding: 12px 14px;
    border: 1px solid var(--gray-border);
    border-radius: 10px;
    font-size: 15px;
    background: #fff;
}

input:focus {
    outline: none;
    border-color: var(--blue);
    box-shadow: 0 0 0 3px rgba(44, 93, 138, 0.12);
}

button,
input[type="submit"] {
    background: var(--navy);
    color: var(--white);
    border: none;
    padding: 12px 18px;
    border-radius: 10px;
    cursor: pointer;
    font-weight: 600;
    transition: 0.2s ease;
}

button:hover,
input[type="submit"]:hover {
    background: var(--blue);
}

.btn-danger {
    background: var(--danger);
}

.btn-danger:hover {
    background: #a93226;
}

.btn-secondary {
    background: #7f8c8d;
}

.btn-secondary:hover {
    background: #677172;
}

.table-wrap {
    overflow-x: auto;
}

.student-table {
    width: 100%;
    border-collapse: collapse;
    margin-top: 10px;
}

.student-table th,
.student-table td {
    padding: 14px;
    border-bottom: 1px solid var(--gray-border);
    text-align: left;
    vertical-align: middle;
}

.student-table th {
    background: var(--light-blue);
    color: var(--navy);
    font-size: 14px;
}

.student-table tr:hover {
    background: #f9fbfd;
}

.inline-form {
    display: inline;
}

.muted {
    color: var(--muted);
    font-size: 14px;
}

.auth-wrapper {
    max-width: 480px;
    margin: 30px auto;
}

.auth-card {
    background: var(--white);
    padding: 28px;
    border-radius: var(--radius);
    box-shadow: var(--shadow);
}

.auth-card h2 {
    margin-top: 0;
    color: var(--navy);
    text-align: center;
}

.auth-card p {
    color: var(--muted);
    text-align: center;
}

.profile-card {
    text-align: center;
}

.profile-image {
    width: 150px;
    height: 150px;
    object-fit: cover;
    border-radius: 50%;
    border: 4px solid var(--blue);
    margin-bottom: 14px;
}

.profile-placeholder {
    width: 150px;
    height: 150px;
    border-radius: 50%;
    background: var(--light-blue);
    color: var(--navy);
    display: flex;
    align-items: center;
    justify-content: center;
    margin: 0 auto 14px;
    font-weight: 600;
}

.hero {
    background: linear-gradient(135deg, #eef4fb, #ffffff);
    border-radius: var(--radius);
    box-shadow: var(--shadow);
    padding: 30px;
}

.hero h2 {
    margin-top: 0;
    color: var(--navy);
    font-size: 32px;
}

.hero p {
    color: var(--muted);
    max-width: 720px;
    line-height: 1.6;
}

.footer-note {
    text-align: center;
    color: var(--muted);
    font-size: 13px;
    margin-top: 30px;
}

small {
    color: var(--muted);
}

@media (max-width: 900px) {
    .layout {
        flex-direction: column;
    }

    .sidebar {
        width: 100%;
        height: auto;
        position: relative;
    }

    .main-area {
        padding: 18px;
    }

    .topbar {
        flex-direction: column;
        align-items: flex-start;
        gap: 10px;
    }
}

Overwriting static/css/style.css


In [ ]:
%%writefile templates/base.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{% block title %}Student Portal{% endblock %}</title>
    <link rel="stylesheet"
          href="{{ url_for('static', filename='css/style.css') }}">
</head>
<body>

<div class="layout">

    <aside class="sidebar">
        <div class="sidebar-brand">
            <img src="{{ url_for('static', filename='images/TUP.png') }}"
                 alt="TUP Logo">
            <h2>Student Portal</h2>
            <p>Modern Academic Dashboard</p>
        </div>

        <div class="sidebar-nav">
            <a href="{{ url_for('home') }}">Home</a>
            <a href="{{ url_for('about') }}">About</a>
            <a href="{{ url_for('contact') }}">Contact</a>

            {% if current_user.is_authenticated %}
                <a href="{{ url_for('students') }}">Students</a>
                <a href="{{ url_for('profile') }}">My Profile</a>

                <form method="POST"
                      action="{{ url_for('logout') }}"
                      class="inline-form">
                    <input type="hidden"
                           name="csrf_token"
                           value="{{ csrf_token() }}">
                    <button type="submit">Logout</button>
                </form>
            {% else %}
                <a href="{{ url_for('login') }}">Login</a>
                <a href="{{ url_for('register') }}">Register</a>
            {% endif %}
        </div>

        {% if current_user.is_authenticated %}
        <div class="sidebar-user">
            <strong>{{ current_user.display_name }}</strong>
            <span>{{ current_user.email }}</span>
        </div>
        {% endif %}
    </aside>

    <section class="main-area">
        <div class="topbar">
            <div>
                <h1>{% block page_heading %}Academic Dashboard{% endblock %}</h1>
                <p>{% block page_subheading %}Flask-based student management system{% endblock %}</p>
            </div>

            {% if current_user.is_authenticated %}
                <span class="role-badge {% if current_user.role == 'admin' %}role-admin{% else %}role-viewer{% endif %}">
                    {{ current_user.role|upper }}
                </span>
            {% endif %}
        </div>

        {% with messages = get_flashed_messages() %}
            {% for msg in messages %}
                <div class="flash">{{ msg }}</div>
            {% endfor %}
        {% endwith %}

        {% block content %}{% endblock %}

        <p class="footer-note">Technological University of the Philippines</p>
    </section>

</div>

</body>
</html>

Overwriting templates/base.html


In [48]:
%%writefile templates/students.html
{% extends 'base.html' %}

{% block title %}Student Dashboard{% endblock %}
{% block page_heading %}Student Management Dashboard{% endblock %}
{% block page_subheading %}Manage and review student records{% endblock %}

{% block content %}

<div class="grid">
    <div class="stat-card">
        <h3>Total Students</h3>
        <p>{{ students|length }}</p>
    </div>

    <div class="stat-card">
        <h3>Current Role</h3>
        <p>{{ current_user.role|capitalize }}</p>
    </div>

    <div class="stat-card">
        <h3>User</h3>
        <p style="font-size:18px;">{{ current_user.display_name }}</p>
    </div>
</div>

<div class="card">
    <h2>Welcome, {{ current_user.display_name }}!</h2>

    {% if current_user.role == 'admin' %}
        <p class="muted">
            You have full access to add, view, and delete student records.
        </p>

        <hr style="margin: 20px 0; border: none; border-top: 1px solid #d8e1ec;">

        <h3>Add New Student</h3>

        <form method="POST" action="{{ url_for('add_student') }}">
            <input type="hidden" name="csrf_token" value="{{ csrf_token() }}">

            <div>
                <label for="name">Full Name</label>
                <input type="text"
                       id="name"
                       name="name"
                       placeholder="Enter student's full name"
                       maxlength="150"
                       required>
            </div>

            <div>
                <label for="email">Email Address</label>
                <input type="email"
                       id="email"
                       name="email"
                       placeholder="Enter student's email"
                       maxlength="150"
                       required>
            </div>

            <button type="submit">Add Student</button>
        </form>

    {% else %}
        <p class="muted">
            You currently have read-only access. You can view student records but cannot add or delete them.
        </p>
    {% endif %}
</div>

<div class="card">
    <h3>Student Records</h3>

    <div class="table-wrap">
        <table class="student-table">
            <thead>
                <tr>
                    <th>ID</th>
                    <th>Full Name</th>
                    <th>Email Address</th>
                    {% if current_user.role == 'admin' %}
                        <th>Action</th>
                    {% endif %}
                </tr>
            </thead>

            <tbody>
                {% for student in students %}
                <tr>
                    <td>{{ student.id }}</td>
                    <td><strong>{{ student.full_name }}</strong></td>
                    <td>{{ student.email }}</td>

                    {% if current_user.role == 'admin' %}
                    <td>
                        <form method="POST"
                              action="{{ url_for('delete_student', id=student.id) }}"
                              class="inline-form">
                            <input type="hidden"
                                   name="csrf_token"
                                   value="{{ csrf_token() }}">
                            <button type="submit" class="btn-danger">Delete</button>
                        </form>
                    </td>
                    {% endif %}
                </tr>
                {% else %}
                <tr>
                    <td colspan="4">No students found.</td>
                </tr>
                {% endfor %}
            </tbody>
        </table>
    </div>
</div>

{% endblock %}

Overwriting templates/students.html


In [49]:
%%writefile templates/profile.html
{% extends 'base.html' %}

{% block title %}My Profile{% endblock %}
{% block page_heading %}Profile Management{% endblock %}
{% block page_subheading %}Update your account details and profile picture{% endblock %}

{% block content %}

<div class="grid">
    <div class="card profile-card">
        {% if current_user.image_filename %}
            <img class="profile-image"
                 src="{{ url_for('static', filename='uploads/' + current_user.image_filename) }}"
                 alt="Profile Picture">
        {% else %}
            <div class="profile-placeholder">No Profile Picture</div>
        {% endif %}

        <h2>{{ current_user.display_name }}</h2>
        <p class="muted">{{ current_user.email }}</p>
        <p>
            <span class="role-badge {% if current_user.role == 'admin' %}role-admin{% else %}role-viewer{% endif %}">
                {{ current_user.role|upper }}
            </span>
        </p>
    </div>

    <div class="card">
        <h3>Update Profile</h3>

        <form method="POST" enctype="multipart/form-data">
            {{ form.hidden_tag() }}

            <div>
                {{ form.display_name.label }}
                {{ form.display_name(size=35, placeholder="Enter display name") }}
                {% for error in form.display_name.errors %}
                    <p class="flash">{{ error }}</p>
                {% endfor %}
            </div>

            <div>
                {{ form.profile_picture.label }}
                {{ form.profile_picture(accept=".jpg,.jpeg,.png") }}
                <small>Allowed formats: JPG, JPEG, PNG. Maximum upload size: 2 MB.</small>
                {% for error in form.profile_picture.errors %}
                    <p class="flash">{{ error }}</p>
                {% endfor %}
            </div>

            <div>
                {{ form.submit() }}
            </div>
        </form>
    </div>
</div>

{% endblock %}

Overwriting templates/profile.html


In [50]:
%%writefile templates/login.html
{% extends 'base.html' %}

{% block title %}Login{% endblock %}
{% block page_heading %}Login{% endblock %}
{% block page_subheading %}Access your student portal account{% endblock %}

{% block content %}

<div class="auth-wrapper">
    <div class="auth-card">
        <h2>Welcome Back</h2>
        <p>Sign in to continue to the academic dashboard.</p>

        <form method="POST">
            {{ form.hidden_tag() }}

            <div>
                {{ form.email.label }}
                {{ form.email(size=30, placeholder="Enter your email") }}
                {% for error in form.email.errors %}
                    <p class="flash">{{ error }}</p>
                {% endfor %}
            </div>

            <div>
                {{ form.password.label }}
                {{ form.password(size=30, placeholder="Enter your password") }}
                {% for error in form.password.errors %}
                    <p class="flash">{{ error }}</p>
                {% endfor %}
            </div>

            <div>
                {{ form.submit() }}
            </div>
        </form>

        <p class="muted">
            Don’t have an account?
            <a href="{{ url_for('register') }}" style="color:#1e3a5f; font-weight:600;">Register here</a>
        </p>
    </div>
</div>

{% endblock %}

Overwriting templates/login.html


In [51]:
%%writefile templates/register.html
{% extends 'base.html' %}

{% block title %}Register{% endblock %}
{% block page_heading %}Register{% endblock %}
{% block page_subheading %}Create your account to access the dashboard{% endblock %}

{% block content %}

<div class="auth-wrapper">
    <div class="auth-card">
        <h2>Create Account</h2>
        <p>Please fill in your account details below.</p>

        <form method="POST">
            {{ form.hidden_tag() }}

            <div>
                {{ form.email.label }}
                {{ form.email(size=30, placeholder="Enter your email") }}
                {% if form.email.description %}
                    <small>{{ form.email.description }}</small>
                {% endif %}
                {% for error in form.email.errors %}
                    <p class="flash">{{ error }}</p>
                {% endfor %}
            </div>

            <div>
                {{ form.password.label }}
                {{ form.password(size=30, placeholder="Create your password") }}
                {% for error in form.password.errors %}
                    <p class="flash">{{ error }}</p>
                {% endfor %}
            </div>

            <div>
                {{ form.submit() }}
            </div>
        </form>

        <p class="muted">
            Already have an account?
            <a href="{{ url_for('login') }}" style="color:#1e3a5f; font-weight:600;">Login here</a>
        </p>
    </div>
</div>

{% endblock %}

Overwriting templates/register.html


In [52]:
%%writefile templates/home.html
{% extends 'base.html' %}

{% block title %}Home{% endblock %}
{% block page_heading %}Home{% endblock %}
{% block page_subheading %}Welcome to the modern academic dashboard{% endblock %}

{% block content %}

<div class="hero">
    <h2>Hello, I am {{ name }}!</h2>
    <p>
        Welcome to my Flask Additional Exercises web application.
        This system demonstrates role-based student management,
        user authentication, profile management, and image upload
        using Flask, SQLAlchemy, and Jinja2 templates.
    </p>
    <p><strong>Section:</strong> {{ section }}</p>
</div>

<div class="grid" style="margin-top: 22px;">
    <div class="stat-card">
        <h3>Core Feature 1</h3>
        <p style="font-size:18px;">Role-Based Access</p>
    </div>

    <div class="stat-card">
        <h3>Core Feature 2</h3>
        <p style="font-size:18px;">Student Management</p>
    </div>

    <div class="stat-card">
        <h3>Core Feature 3</h3>
        <p style="font-size:18px;">Profile Management</p>
    </div>
</div>

{% endblock %}

Overwriting templates/home.html
